# كشكول التجارب: دمج البحث الدلالي والبحث الحرفي (المحاضرة 41)

أهلاً بيك! هنا في الكشكول ده هنجرب مع بعض فكرة بسيطة جداً بس هي أساس كل حاجة في البحث الهجين (`Hybrid Search`).

الفكرة ببساطة:
- أحياناً لما بتدور على معلومة، بتكتب نفس الكلمة بالظبط (زي كود أو اسم برنامج).
- وأحياناً تانية بتكتب معنى مشابه للكلمة من غير ما تستخدم نفس الحروف.

الكمبيوتر زمان كان بيفهم الحروف بالظبط (`TF-IDF`).  
والنماذج الحديثة بتفهم المعنى الدلالي (`Embeddings`).  
طب ليه منجمعش الاتنين مع بعض بمعادلة سهلة وبسيطة؟ تعال نجرب!


In [1]:
# أول حاجة: نستورد كام مكتبة خفيفة نحتاجهم
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("المكتبات جاهزة، يلا بينا!")


المكتبات جاهزة، يلا بينا!


### 1. شوية جمل هنجرب عليهم
تعال نكتب 3 جمل عن الـ RAG، وجملة تانية بعيدة تماماً عن الموضوع علشان نشوف هل النظام هيعرف يفرق ولا لأ:


In [2]:
# دي الوثائق اللي هنجرب نبحث فيها
documents = [
    "LangChain is a framework for building LLM applications",
    "RAG systems retrieve relevant context from vector stores",
    "Vector embeddings capture the semantic meaning of text",
    "The weather in Paris is sunny and warm today"  # دي جملة بعيدة خالص للتجربة
]

# وده السؤال أو البحث اللي عايزين نعمله
query = "building AI apps using language models"

print(f"السؤال بتاعنا: '{query}'\n")
for i, doc in enumerate(documents, 1):
    print(f"الجملة رقم {i}: {doc}")


السؤال بتاعنا: 'building AI apps using language models'

الجملة رقم 1: LangChain is a framework for building LLM applications
الجملة رقم 2: RAG systems retrieve relevant context from vector stores
الجملة رقم 3: Vector embeddings capture the semantic meaning of text
الجملة رقم 4: The weather in Paris is sunny and warm today


### 2. نحسب البحث الحرفي (TF-IDF)
هنا الكمبيوتر هيبص على الحروف والكلمات المشتركة:


In [3]:
# بنحسب الـ TF-IDF لكل جملة
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(documents)
query_tfidf = tfidf.transform([query])

# نحسب التشابه
sparse_scores = cosine_similarity(query_tfidf, tfidf_matrix).flatten()

print("نتائج البحث الحرفي (TF-IDF):")
for i, score in enumerate(sparse_scores, 1):
    print(f"  الجملة {i}: {score:.3f}")


نتائج البحث الحرفي (TF-IDF):
  الجملة 1: 0.389
  الجملة 2: 0.000
  الجملة 3: 0.000
  الجملة 4: 0.000


### 3. نحسب البحث الدلالي (Dense Embeddings)
هنا بقى الموديل هيفهم المعنى حتى لو الكلمات مختلفة (مثلاً: `AI apps` شبه `LLM applications`):


In [4]:
# بنحمل موديل صغير وسريع يفهم المعنى
dense_model = SentenceTransformer("all-MiniLM-L6-v2")

doc_embeddings = dense_model.encode(documents)
query_embedding = dense_model.encode([query])

dense_scores = cosine_similarity(query_embedding, doc_embeddings).flatten()

print("نتائج البحث الدلالي (Embeddings):")
for i, score in enumerate(dense_scores, 1):
    print(f"  الجملة {i}: {score:.3f}")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

نتائج البحث الدلالي (Embeddings):
  الجملة 1: 0.352
  الجملة 2: 0.089
  الجملة 3: 0.243
  الجملة 4: 0.039


### 4. تعال ندمج الاتنين ونشوف السحر!
المعادلة بسيطة جداً:  
`الدرجة النهائية = (alpha * الدلالي) + ((1 - alpha) * الحرفي)`  
قيمة `alpha` هي نسبة التركيز على المعنى الدلالي. مثلاً لو خليناها `0.5` يبقى 50% دلالي و 50% حرفي.


In [5]:
# جرب تغير الـ alpha دي وشوف النتيجة
alpha = 0.5

# المعادلة الهجينة
hybrid_scores = (alpha * dense_scores) + ((1 - alpha) * sparse_scores)

# نرتب الجمل من الأعلى للأقل
ranked_indices = np.argsort(hybrid_scores)[::-1]

print(f"الترتيب النهائي للنتائج (مع alpha = {alpha}):\n")
for rank, idx in enumerate(ranked_indices, 1):
    print(f"المركز {rank}: [درجة: {hybrid_scores[idx]:.3f}] {documents[idx]}")


الترتيب النهائي للنتائج (مع alpha = 0.5):

المركز 1: [درجة: 0.370] LangChain is a framework for building LLM applications
المركز 2: [درجة: 0.122] Vector embeddings capture the semantic meaning of text
المركز 3: [درجة: 0.044] RAG systems retrieve relevant context from vector stores
المركز 4: [درجة: 0.020] The weather in Paris is sunny and warm today


---
### مساحتك للتجربة الحرة
جرب في الخلية دي تغير جملة الـ `query` أو تغير قيمة `alpha` لـ `0.9` (معنى أكتر) أو `0.1` (حروف أكتر)، وشوف إيه اللي هيحصل في ترتيب الجمل!


In [6]:
# جرب هنا براحتك:
my_query = "sunny weather forecast"
my_alpha = 0.7

# احسب وشوف النتيجة بنفسك!
